# Tract group selection

Port of `packages/niivue/examples/tract.group.html`. Loads the Yeh 2022 TRX tract atlas and uses a Jupyter dropdown to isolate selected tract groups.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"

group_names = [
    "AC",
    "AF_L",
    "AF_R",
    "CC",
    "CST_L",
    "CST_R",
    "IFOF_L",
    "IFOF_R",
    "ILF_L",
    "ILF_R",
    "SLF1_L",
    "SLF1_R",
    "SLF2_L",
    "SLF2_R",
    "TR_S_L",
    "TR_S_R",
    "UF_L",
    "UF_R",
    "VOF_L",
    "VOF_R",
]

group_palette = [
    [230, 25, 75, 255],
    [60, 180, 75, 255],
    [0, 130, 200, 255],
    [255, 225, 25, 255],
]

nv = NiiVue(background_color=[1, 1, 1, 1], slice_type=4, backend="webgl2")

slice_type = widgets.Dropdown(options=[("Render", 4), ("A+C+S+R", 3)], value=4, description="Slice")
color_mode = widgets.Dropdown(
    options=[("Local direction", "local"), ("Global direction", "global"), ("Fixed", "fixed")],
    value="local",
    description="Color",
)
group_select = widgets.Dropdown(
    options=[("All", "")] + [(name, name) for name in group_names], value="", description="Group"
)
decimation = widgets.IntSlider(value=1, min=1, max=20, step=1, description="Decimation")


def update_slice(change):
    nv.slice_type = change["new"]


def update_decimation(change):
    nv.set_tract_options(0, {"decimation": change["new"]})


def apply_color(change=None):
    group = group_select.value
    if group:
        nv.set_tract_options(
            0, {"groupColors": {group: group_palette[0]}, "colorBy": color_mode.value}
        )
    else:
        nv.set_tract_options(0, {"groupColors": None, "colorBy": color_mode.value})


slice_type.observe(update_slice, names="value")
decimation.observe(update_decimation, names="value")
color_mode.observe(apply_color, names="value")
group_select.observe(apply_color, names="value")

controls = widgets.VBox([widgets.HBox([slice_type, color_mode, group_select]), decimation])
display(controls)
display(nv)

nv.set_clip_planes([[0.1, 180, 20], [0.1, 0, -20]])
nv.load_volumes([{"url": f"{VOLUMES}/mni152.nii.gz"}])
nv.load_meshes([{"url": f"{MESHES}/yeh2022.trx", "rgba255": [0, 142, 200, 255]}])